# Train Graph of Graphs in Colab

This notebook clones `richie955/graphofgraphs` into the Colab runtime, loads the PrimeKG dataset from the repository, and trains the GoG model. It does not use Google Drive. Runtime files are temporary; download reports/checkpoints before disconnecting if you need to keep them.

In [ ]:
# Select Runtime > Change runtime type > T4 GPU for GPU training.
from pathlib import Path
import os, subprocess, sys

REPO_URL = "https://github.com/richie955/graphofgraphs.git"
REPO_DIR = Path("/content/graphofgraphs")
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
os.chdir(REPO_DIR)
print("Repository:", REPO_DIR)
print("Dataset:", REPO_DIR / "docs/primekg_gog_30000.json")
print("GPU available:", subprocess.run([sys.executable, "-c", "import torch; print(torch.cuda.is_available())"], check=True, capture_output=True, text=True).stdout.strip())

In [ ]:
# Colab normally includes PyTorch. Install it only if this runtime does not.
try:
    import torch
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "torch"], check=True)
    import torch

dataset = REPO_DIR / "docs/primekg_gog_30000.json"
assert dataset.is_file(), f"Dataset not found: {dataset}"
print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
print("Dataset size (MB):", round(dataset.stat().st_size / 1024**2, 1))

In [ ]:
# Train GoG. Lower --batch-size if Colab runs out of GPU memory.
!python train_stage9.py docs/primekg_gog_30000.json --model gog --device auto --output-dir runs/colab-primekg --epochs 100 --patience 15 --batch-size 1024

In [ ]:
# Package the report and checkpoint for download before the runtime ends.
import shutil
from google.colab import files
result_dir = REPO_DIR / "runs/colab-primekg"
archive = shutil.make_archive("graphofgraphs-colab-results", "zip", result_dir)
files.download(archive)